# Single-threshold directional change: comparing HMM emission families

This notebook rebuilds the one-threshold event ledger from `1_threshold_dc_explore.ipynb`, then compares six emission families for a two-state, filtered, event-time HMM: which distribution describes completed-event R best out of sample, which states survive a change of family, and how stable that answer is across folds and seeds.

For each completed extreme-to-extreme trend:

$$\mathrm{TMV} = \frac{|P_{\mathrm{EXT},n} - P_{\mathrm{EXT},n-1}|}{P_{\mathrm{EXT},n-1}\,\theta}$$

$$T = t_{\mathrm{EXT},n} - t_{\mathrm{EXT},n-1}$$

$$R = \frac{\mathrm{TMV}\,\theta}{T}$$

The ending extreme is identified retrospectively. Therefore, TMV, T, and R are not available at `ext_end_idx`; they first become knowable at the later DC confirmation, `dcc_idx`. The notebook records that timestamp as `information_time`.

In [1]:
# Reproducibility: pin thread counts before NumPy or PyTorch is imported, so EM sums run in a fixed order.
import os

HMM_THREADS = 1
for _variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_variable] = str(HMM_THREADS)

from dataclasses import dataclass, field
from enum import StrEnum

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

from dc_basics import prepare_event_ledger
from dc_basics.core import parse_dc_events
from dc_basics.data import resolve_data_root
from dc_basics.hmm import DistType, InferenceMode
from dc_basics.hmm_support import (
    SEMANTIC_R_STATE_NAMES,
    configure_reproducible_threads,
    create_hmm_instance,
    raw_r_predictive_loglik,
    semantic_r_state_order,
)
from dc_basics.plotting import (
    plot_dc_indicator_distributions,
    plot_directional_change,
    plot_hmm_label_agreement,
    plot_hmm_model_event_strips,
    plot_hmm_model_label_shares,
    plot_hmm_model_price_comparison,
)

print(configure_reproducible_threads(HMM_THREADS))

{'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1', 'NUMEXPR_NUM_THREADS': '1', 'torch_num_threads': 1}


## Configuration

In [2]:
DATA_ROOT = resolve_data_root()  # $DC_DATA_ROOT or the git-ignored .dc_data_root file
SYMBOL = "US500.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"

THETA = 0.003
N_PRICES = 5_000

print(f"Symbol: {SYMBOL}")
print(f"Data: {DATA_PATH}")
print(f"Threshold: {THETA:.4%}")
print(f"Chart window: {N_PRICES:,} prices")

Symbol: US500.r
Data: D:\data_dump\market_data\raw\FXPIG-Server\5\US500.r.parquet
Threshold: 0.3000%
Chart window: 5,000 prices


## Load and audit the bars

Directional-change parsing is path-dependent, so the price series must be chronological, unique, finite, and strictly positive. The cell below refuses unsorted, duplicated, non-finite, or non-positive prices rather than silently repairing them. If `time` is stored as a column rather than the Parquet index, it is promoted to the index explicitly.

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))

raw = raw.sort_index()
if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no 'close' column.")

prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all():
    raise ValueError("Close contains NaN or infinite values.")
if prices.le(0.0).any():
    raise ValueError("DC thresholds require strictly positive prices.")

print(f"Bars: {len(prices):,}")
print(f"Start: {prices.index[0]}")
print(f"End: {prices.index[-1]}")
print(f"Timezone: {prices.index.tz}")
raw.tail()

Bars: 245,467
Start: 2023-05-22 14:00:00+02:00
End: 2026-11-09 13:30:00+02:00
Timezone: pytz.FixedOffset(120)


,open,high,low,close,tick_volume,spread
date,,,,,,
2026-11-09 13:10:00+02:00,7425.52,7427.97,7420.02,7420.52,1099.0,41
2026-11-09 13:15:00+02:00,7420.52,7440.15,7418.32,7432.52,1494.0,41
2026-11-09 13:20:00+02:00,7432.52,7436.14,7427.27,7429.02,1296.0,41
2026-11-09 13:25:00+02:00,7429.27,7433.14,7427.52,7429.90,857.0,41
2026-11-09 13:30:00+02:00,7429.90,7433.77,7429.52,7432.27,677.0,41


## Calculate completed directional-change events

`parse_dc_events` returns one row per completed trend. `prepare_event_ledger` then attaches the event identity and `information_time = dcc_idx`, because that confirmation bar is when the ending extreme and its completed TMV, T, and R become knowable. It also checks the ledger's causal invariants: no two events share a confirmation bar, no ending extreme lies after its information time, and $R = \mathrm{TMV}\,\theta/T$ on every row.

The ledger states direction explicitly. `direction` (identical to `completed_leg_direction`) is the leg that **ended**. `dcc_direction` is the direction of the confirmation itself, and `new_live_mode` is the mode that is live from `information_time` on. A completed `up` leg is confirmed by a downward DCC and switches the live mode down.

The library's string ID is kept as `ledger_event_id` (for example `E17`). This lab also keeps the plain event number as `event_id`, because its plots place events on a numeric event-order axis.

In [4]:
source_events = parse_dc_events(prices=prices, theta=THETA)
if source_events.empty:
    raise ValueError("No completed DC events were found. Check the data and threshold.")

events = prepare_event_ledger(source_events, prefix="E", scale="single", theta=THETA)
events = events.rename(columns={"event_id": "ledger_event_id"})
events.insert(0, "event_id", np.arange(1, len(events) + 1, dtype=np.int64))

event_columns = [
    "event_id", "threshold", "direction", "dcc_direction", "new_live_mode",
    "ext_start_idx", "ext_start_price", "ext_end_idx", "ext_end_price", "dcc_idx", "dcc_price",
    "information_time", "tmv", "t", "r",
]

print(f"Completed events: {len(events):,}")
events[event_columns].tail(10)

Completed events: 3,862

,event_id,threshold,direction,dcc_direction,new_live_mode,ext_start_idx,ext_start_price,ext_end_idx,ext_end_price,dcc_idx,dcc_price,information_time,tmv,t,r
3852,3853,0.003,up,down,down,2026-11-06 09:30:00+02:00,7335.90,2026-11-06 11:35:00+02:00,7401.27,2026-11-06 12:05:00+02:00,7378.02,2026-11-06 12:05:00+02:00,2.970324,25,0.000356
3853,3854,0.003,down,up,up,2026-11-06 11:35:00+02:00,7401.27,2026-11-06 15:25:00+02:00,7357.72,2026-11-06 16:35:00+02:00,7395.22,2026-11-06 16:35:00+02:00,1.961375,46,0.000128
3854,3855,0.003,up,down,down,2026-11-06 15:25:00+02:00,7357.72,2026-11-06 17:10:00+02:00,7433.97,2026-11-06 17:30:00+02:00,7397.72,2026-11-06 17:30:00+02:00,3.454422,21,0.000493
3855,3856,0.003,down,up,up,2026-11-06 17:10:00+02:00,7433.97,2026-11-06 17:30:00+02:00,7397.72,2026-11-06 17:45:00+02:00,7426.82,2026-11-06 17:45:00+02:00,1.625421,4,0.001219
3856,3857,0.003,up,down,down,2026-11-06 17:30:00+02:00,7397.72,2026-11-06 18:50:00+02:00,7431.07,2026-11-06 19:35:00+02:00,7392.82,2026-11-06 19:35:00+02:00,1.502715,16,0.000282
3857,3858,0.003,down,up,up,2026-11-06 18:50:00+02:00,7431.07,2026-11-06 19:50:00+02:00,7384.32,2026-11-06 20:30:00+02:00,7413.62,2026-11-06 20:30:00+02:00,2.097051,12,0.000524
3858,3859,0.003,up,down,down,2026-11-06 19:50:00+02:00,7384.32,2026-11-06 22:05:00+02:00,7414.12,2026-11-06 22:50:00+02:00,7378.12,2026-11-06 22:50:00+02:00,1.345193,27,0.000149
3859,3860,0.003,down,up,up,2026-11-06 22:05:00+02:00,7414.12,2026-11-09 01:55:00+02:00,7374.65,2026-11-09 02:55:00+02:00,7402.14,2026-11-09 02:55:00+02:00,1.774542,34,0.000157
3860,3861,0.003,up,down,down,2026-11-09 01:55:00+02:00,7374.65,2026-11-09 06:45:00+02:00,7432.15,2026-11-09 07:50:00+02:00,7409.40,2026-11-09 07:50:00+02:00,2.598993,58,0.000134
3861,3862,0.003,down,up,up,2026-11-09 06:45:00+02:00,7432.15,2026-11-09 09:50:00+02:00,7373.90,2026-11-09 10:20:00+02:00,7402.27,2026-11-09 10:20:00+02:00,2.612524,37,0.000212


## Visualize the last N prices

The close is shown with completed extreme-to-extreme trends and their later DC confirmations. A turning-point marker describes where an extreme occurred; its hover text shows when that extreme became knowable. The final unfinished event is intentionally absent because `parse_dc_events` emits completed events only.

In [5]:
fig = plot_directional_change(prices=prices, events=events, n_prices=N_PRICES, title=f"{SYMBOL} | theta={THETA:.4%}",)
fig.show()

## Descriptive EDA of TMV, T, and R

The mean answers the average question, but these event indicators are typically right-skewed. Median and tail quantiles are reported beside it so a small number of extreme events cannot silently determine the interpretation. These are descriptive event-level results: consecutive DC events are path-dependent, so the rows must not be treated as independent observations for significance testing.

In [6]:
INDICATORS = ["tmv", "t", "r"]
QUANTILES = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

overall_summary = events[INDICATORS].describe(percentiles=QUANTILES).T
overall_summary.index.name = "indicator"
overall_summary.style.format(precision=6)

,count,mean,std,min,10%,25%,50%,75%,90%,95%,99%,max
indicator,,,,,,,,,,,,
tmv,3862.000000,2.398213,1.552848,1.000390,1.130821,1.373348,1.912538,2.897117,4.209480,5.253013,8.067023,21.481858
t,3862.000000,63.547385,90.439018,1.000000,5.000000,11.000000,28.000000,75.000000,179.900000,240.000000,415.340000,1100.000000
r,3862.000000,0.000509,0.000920,0.000009,0.000044,0.000085,0.000214,0.000502,0.001215,0.002026,0.004531,0.011954


### Direction-conditioned averages

The event count must be read with every directional comparison. A difference between up and down averages is descriptive here; it is not yet evidence of a stable or tradable asymmetry.

In [7]:
direction_summary = events.groupby("direction", sort=False).agg(
    event_count=("event_id", "size"),
    tmv_mean=("tmv", "mean"),
    tmv_median=("tmv", "median"),
    t_mean=("t", "mean"),
    t_median=("t", "median"),
    r_mean=("r", "mean"),
    r_median=("r", "median"),
)
direction_summary.style.format({
    "event_count": "{:,}",
    "tmv_mean": "{:.6f}",
    "tmv_median": "{:.6f}",
    "t_mean": "{:.2f}",
    "t_median": "{:.2f}",
    "r_mean": "{:.6e}",
    "r_median": "{:.6e}",
})

,event_count,tmv_mean,tmv_median,t_mean,t_median,r_mean,r_median
direction,,,,,,,
up,"1,931",2.459193,1.931890,73.58,29.00,5.020026e-04,2.003685e-04
down,"1,931",2.337233,1.893983,53.51,26.00,5.156014e-04,2.323405e-04


### Direction-conditioned distributions

Each histogram is normalized within direction, so up and down shapes remain comparable even if their event counts differ. The initial x-axis view ends at the full-sample 99th percentile; Plotly's **Autoscale** button reveals all tail observations.

In [8]:
distribution_fig = plot_dc_indicator_distributions(events)
distribution_fig.show()

## Find events whose extremes are consecutive observations

`T = 1` means the starting and ending extremes occupy adjacent rows in the stored close series. `elapsed_minutes` checks whether those observations are also one 5-minute period apart in wall-clock time. `confirmation_delay_bars` measures how many subsequent observations were needed to confirm the ending extreme.

In [9]:
consecutive_events = events.loc[events["t"] == 1].copy()
consecutive_events["elapsed_minutes"] = (consecutive_events["ext_end_idx"] - consecutive_events["ext_start_idx"]).dt.total_seconds() / 60.0
consecutive_events["observed_move_pct"] = (consecutive_events["ext_end_price"] / consecutive_events["ext_start_price"] - 1.0) * 100.0
consecutive_events["confirmation_delay_bars"] = (consecutive_events["dcc_pos"] - consecutive_events["ext_end_pos"])

consecutive_columns = [
    "event_id", "direction", "ext_start_idx", "ext_start_price",
    "ext_end_idx", "ext_end_price", "elapsed_minutes", "observed_move_pct",
    "tmv", "r", "dcc_idx", "confirmation_delay_bars",
]

print(f"Events with T=1: {len(consecutive_events):,} of {len(events):,}")
consecutive_events[consecutive_columns]

Events with T=1: 63 of 3,862


,event_id,direction,ext_start_idx,ext_start_price,ext_end_idx,ext_end_price,elapsed_minutes,observed_move_pct,tmv,r,dcc_idx,confirmation_delay_bars
617,618,down,2024-02-02 19:35:00+02:00,4960.44,2024-02-02 19:40:00+02:00,4941.44,5.0,-0.383031,1.276768,0.003830,2024-02-02 20:15:00+02:00,7
691,692,down,2024-03-20 21:25:00+02:00,5202.49,2024-03-20 21:30:00+02:00,5186.14,5.0,-0.314273,1.047575,0.003143,2024-03-20 21:40:00+02:00,2
923,924,down,2024-07-19 00:45:00+02:00,5562.82,2024-07-19 00:50:00+02:00,5544.77,5.0,-0.324476,1.081586,0.003245,2024-07-19 03:05:00+02:00,15
992,993,up,2024-08-02 18:30:00+02:00,5358.32,2024-08-02 18:35:00+02:00,5381.32,5.0,0.429239,1.430797,0.004292,2024-08-02 18:55:00+02:00,4
1017,1018,down,2024-08-05 18:35:00+02:00,5174.47,2024-08-05 18:40:00+02:00,5158.92,5.0,-0.300514,1.001713,0.003005,2024-08-05 18:45:00+02:00,1
...,...,...,...,...,...,...,...,...,...,...,...,...
3452,3453,up,2026-06-11 19:15:00+02:00,7293.39,2026-06-11 19:20:00+02:00,7315.37,5.0,0.301369,1.004563,0.003014,2026-06-11 19:55:00+02:00,7
3818,3819,up,2026-10-29 22:30:00+02:00,7689.72,2026-10-29 22:35:00+02:00,7715.72,5.0,0.338114,1.127046,0.003381,2026-10-29 22:40:00+02:00,1
3819,3820,down,2026-10-29 22:35:00+02:00,7715.72,2026-10-29 22:40:00+02:00,7688.97,5.0,-0.346695,1.155649,0.003467,2026-10-29 22:45:00+02:00,1
3838,3839,up,2026-11-05 17:50:00+02:00,7424.32,2026-11-05 17:55:00+02:00,7448.57,5.0,0.326629,1.088764,0.003266,2026-11-05 18:00:00+02:00,1


### Inspect one consecutive-extreme event on the raw bars

Change `EVENT_ID` to inspect another row from the table above. The displayed path begins at the starting extreme and ends at the later confirmation bar.

In [10]:
if consecutive_events.empty:
    raise ValueError("No T=1 events are available at the current threshold.")

EVENT_ID = int(consecutive_events.iloc[-1]["event_id"])
selected_event = events.loc[events["event_id"] == EVENT_ID].iloc[0]

event_bars = raw.iloc[
    int(selected_event["ext_start_pos"]):int(selected_event["dcc_pos"]) + 1
].copy()
event_bars["dc_landmark"] = ""
event_bars.loc[selected_event["ext_start_idx"], "dc_landmark"] = "start EXT"
event_bars.loc[selected_event["ext_end_idx"], "dc_landmark"] = "end EXT"
event_bars.loc[selected_event["dcc_idx"], "dc_landmark"] = "DCC confirmation"

print(
    f"Event {EVENT_ID}: {selected_event['direction']} | "
    f"T={selected_event['t']} | TMV={selected_event['tmv']:.4f} | R={selected_event['r']:.3e}"
)
event_bars[["open", "high", "low", "close", "dc_landmark"]]

Event 3840: down | T=1 | TMV=1.0517 | R=3.155e-03


,open,high,low,close,dc_landmark
date,,,,,
2026-11-05 17:55:00+02:00,7424.44,7455.82,7417.57,7448.57,start EXT
2026-11-05 18:00:00+02:00,7448.07,7451.07,7423.62,7425.07,end EXT
2026-11-05 18:05:00+02:00,7425.07,7449.77,7418.70,7447.32,
2026-11-05 18:10:00+02:00,7446.32,7476.32,7445.82,7473.07,DCC confirmation


## Competing HMM emission distributions

We now compare six two-state HMMs while keeping the event series, chronological 80/20 split, causal filtering, restart count, and semantic state definition fixed.

The observation scale follows each distribution's support:

- Normal, Student-t, and Lambda use `log(R)` because they have real-valued support.
- LogNormal, Gamma, and Exponential use raw positive `R`.

For a fair likelihood comparison, every score is put on the raw-`R` density scale. A density over $\log R$ is converted with the Jacobian: $\log p_R(r) = \log p_{\log R}(\log r) - \log r$. That applies to the three log-R candidates **and** to LogNormal: pomegranate's `LogNormal.log_probability` evaluates the Normal density of $\log R$ and omits the $-\log R$ term. `dc_basics.hmm_support.raw_r_predictive_loglik` applies the correction to both. After it, Normal-on-log-R and LogNormal-on-raw-R are the same model and must score the same; the notebook asserts this as an implementation check. (Earlier runs scored LogNormal without the Jacobian, which put it last by exactly the mean of $-\log R$.)

LogNormal emissions are built with `create_hmm_instance`, which starts them from seeded k-means on $\log R$ and honours `covariance_type` and `min_cov`. The vendored builder started every LogNormal state from an unseeded random point near $\log R = 0$, while the data sit near $\log R \approx -9$. That made the candidate irreproducible from run to run and sometimes collapsed it to one state.

The primary comparison is chronological out-of-sample predictive log-likelihood. Label balance is descriptive, not a selection target. The sequence-order premium compares the real OOS ordering with shuffled versions of the same OOS values; a positive premium indicates that the transition dynamics explain ordering beyond merely partitioning the marginal tail. The fold-and-seed table at the end of the notebook shows how much of any ranking survives other splits and seeds.

In [11]:
class ObservationTransform(StrEnum):
    RAW_R = "raw_R"
    LOG_R = "log_R"


class CandidateName(StrEnum):
    NORMAL_LOG_R = "normal_log_R"
    STUDENTT_LOG_R = "studentt_log_R"
    LAMBDA_LOG_R = "lambda_log_R"
    LOGNORMAL_RAW_R = "lognormal_raw_R"
    GAMMA_RAW_R = "gamma_raw_R"
    EXPONENTIAL_RAW_R = "exponential_raw_R"


@dataclass(frozen=True)
class HmmCandidateSpec:
    name: CandidateName
    distribution: DistType
    transform: ObservationTransform
    kwargs: dict[str, object] = field(default_factory=dict)


N_STATES = 2
TRAIN_FRACTION = 0.80
RANDOM_STATE = 100
MAX_ITER = 500
N_RESTARTS = 5
N_SEQUENCE_SHUFFLES = 100
COMPARISON_PRICE_BARS = 20_000

# Normal(log R) and LogNormal(R) are the same model on the raw-R scale; their OOS scores must agree.
JACOBIAN_CHECK_TOLERANCE = 1e-3

# Stability grid: expanding chronological folds (train on the first fraction, score the next block)
# and extra seeds on the main split. Each entry costs one fit per candidate.
RUN_STABILITY_GRID = True
FOLD_TRAIN_FRACTIONS = (0.60, 0.70, TRAIN_FRACTION)
EXTRA_STABILITY_SEEDS = (RANDOM_STATE + 1, RANDOM_STATE + 2)

CANDIDATES = (
    HmmCandidateSpec(CandidateName.NORMAL_LOG_R, DistType.NORMAL, ObservationTransform.LOG_R,
                     {"covariance_type": "diag", "min_cov": 1e-6}),
    HmmCandidateSpec(CandidateName.STUDENTT_LOG_R, DistType.STUDENTT, ObservationTransform.LOG_R,
                     {"covariance_type": "diag", "min_cov": 1e-6, "dofs": 5}),
    HmmCandidateSpec(CandidateName.LAMBDA_LOG_R, DistType.LAMDA, ObservationTransform.LOG_R),
    HmmCandidateSpec(CandidateName.LOGNORMAL_RAW_R, DistType.LOGNORMAL, ObservationTransform.RAW_R,
                     {"covariance_type": "diag", "min_cov": 1e-6}),
    HmmCandidateSpec(CandidateName.GAMMA_RAW_R, DistType.GAMMA, ObservationTransform.RAW_R),
    HmmCandidateSpec(CandidateName.EXPONENTIAL_RAW_R, DistType.EXPONENTIAL, ObservationTransform.RAW_R),
)

In [12]:
comparison_events = events.copy().sort_values("information_time").reset_index(drop=True)
if comparison_events["r"].isna().any() or not np.isfinite(comparison_events["r"]).all():
    raise ValueError("R contains NaN or infinite observations.")
if (comparison_events["r"] <= 0).any():
    raise ValueError("All candidate distributions require strictly positive R before transformation.")

comparison_events["log_r"] = np.log(comparison_events["r"])
r_all = comparison_events[["r"]].to_numpy(dtype=float)
log_r_all = comparison_events[["log_r"]].to_numpy(dtype=float)
train_event_count = int(len(comparison_events) * TRAIN_FRACTION)
if train_event_count < N_STATES or train_event_count >= len(comparison_events):
    raise ValueError("The chronological split leaves too few train or OOS events.")

event_sample = np.where(np.arange(len(comparison_events)) < train_event_count, "train", "oos")
print(f"Events: {len(comparison_events):,} | train: {train_event_count:,} | OOS: {len(comparison_events) - train_event_count:,}")
print(f"Train ends: {comparison_events.loc[train_event_count - 1, 'information_time']}")
print(f"OOS starts: {comparison_events.loc[train_event_count, 'information_time']}")

Events: 3,862 | train: 3,089 | OOS: 773
Train ends: 2026-03-10 08:40:00+02:00
OOS starts: 2026-03-10 10:30:00+02:00


In [13]:
def observations_for(spec: HmmCandidateSpec) -> np.ndarray:
    return log_r_all if spec.transform == ObservationTransform.LOG_R else r_all


def fit_candidate(spec: HmmCandidateSpec, train_count: int, seed: int):
    model = create_hmm_instance(
        dist_type=spec.distribution, n_states=N_STATES, random_state=seed,
        max_iter=MAX_ITER, inference_mode=InferenceMode.FILTERING, n_restarts=N_RESTARTS, **spec.kwargs
    )
    model.fit(observations_for(spec)[:train_count])
    return model


def candidate_raw_r_scores(model, spec: HmmCandidateSpec, observations: np.ndarray, raw_r: np.ndarray) -> np.ndarray:
    # Raw-R predictive log-density, with the -log R Jacobian for log-R fits and for pomegranate's LogNormal.
    return raw_r_predictive_loglik(
        model, observations, raw_r, observations_are_log_r=spec.transform == ObservationTransform.LOG_R
    )


def semantic_fast_flags(probabilities: np.ndarray, train_count: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Shared rule: slow_R has the lower filtered-probability-weighted training mean of raw R.
    order, centers = semantic_r_state_order(probabilities[:train_count], r_all[:train_count].reshape(-1))
    semantic_probabilities = probabilities[:, order]
    return order, centers, semantic_probabilities


def flip_rate(labels: np.ndarray) -> float:
    return float(np.mean(labels[1:] != labels[:-1])) if len(labels) > 1 else np.nan

In [14]:
comparison_models = {}
comparison_label_frames = []
comparison_result_rows = []

for spec in CANDIDATES:
    model_name = spec.name.value
    try:
        observations = observations_for(spec)
        train_observations = observations[:train_event_count]
        model = fit_candidate(spec, train_event_count, RANDOM_STATE)

        filtered_probabilities = model.predict_proba(observations)
        semantic_order, semantic_centers, semantic_probabilities = semantic_fast_flags(
            filtered_probabilities, train_event_count
        )
        semantic_state_index = np.argmax(semantic_probabilities, axis=1)
        semantic_labels = np.array(SEMANTIC_R_STATE_NAMES, dtype=object)[semantic_state_index]
        state_confidence = semantic_probabilities.max(axis=1)

        raw_scale_scores = candidate_raw_r_scores(model, spec, observations, r_all)
        train_scores = raw_scale_scores[:train_event_count]
        oos_scores = raw_scale_scores[train_event_count:]

        rng = np.random.default_rng(RANDOM_STATE)
        shuffled_oos_means = []
        for _ in range(N_SEQUENCE_SHUFFLES):
            permutation = rng.permutation(len(observations) - train_event_count)
            shuffled_observations = np.vstack(
                [train_observations, observations[train_event_count:][permutation]]
            )
            shuffled_raw_r = np.vstack([r_all[:train_event_count], r_all[train_event_count:][permutation]])
            shuffled_scores = candidate_raw_r_scores(model, spec, shuffled_observations, shuffled_raw_r)
            shuffled_oos_means.append(float(shuffled_scores[train_event_count:].mean()))

        transition_matrix = model.transition_prob()[np.ix_(semantic_order, semantic_order)]
        train_labels = semantic_labels[:train_event_count]
        oos_labels = semantic_labels[train_event_count:]
        train_fast_share = float(np.mean(train_labels == "fast_R"))
        oos_fast_share = float(np.mean(oos_labels == "fast_R"))
        collapsed = train_fast_share in {0.0, 1.0} or oos_fast_share in {0.0, 1.0}

        comparison_models[model_name] = model
        comparison_label_frames.append(pd.DataFrame({
            "event_id": comparison_events["event_id"].to_numpy(),
            "information_time": comparison_events["information_time"].to_numpy(),
            "sample": event_sample,
            "model": model_name,
            "state_name": semantic_labels,
            "state_probability": state_confidence,
            "p_fast_r": semantic_probabilities[:, 1],
        }))
        comparison_result_rows.append({
            "model": model_name,
            "distribution": spec.distribution.name,
            "observation": spec.transform.value,
            "status": "collapsed" if collapsed else "ok",
            "train_mean_raw_R_predictive_ll": float(train_scores.mean()),
            "oos_mean_raw_R_predictive_ll": float(oos_scores.mean()),
            "oos_p05_raw_R_predictive_ll": float(np.quantile(oos_scores, 0.05)),
            "oos_sequence_order_premium": float(oos_scores.mean() - np.median(shuffled_oos_means)),
            "slow_R_center": float(semantic_centers[0]),
            "fast_R_center": float(semantic_centers[1]),
            "center_ratio": float(semantic_centers[1] / semantic_centers[0]),
            "train_fast_share": train_fast_share,
            "oos_fast_share": oos_fast_share,
            "train_mean_confidence": float(state_confidence[:train_event_count].mean()),
            "oos_mean_confidence": float(state_confidence[train_event_count:].mean()),
            "train_flip_rate": flip_rate(train_labels),
            "oos_flip_rate": flip_rate(oos_labels),
            "slow_R_self_transition": float(transition_matrix[0, 0]),
            "fast_R_self_transition": float(transition_matrix[1, 1]),
            "error": "",
        })
    except Exception as exc:
        comparison_result_rows.append({
            "model": model_name, "distribution": spec.distribution.name,
            "observation": spec.transform.value, "status": "failed",
            "oos_mean_raw_R_predictive_ll": np.nan, "error": repr(exc)
        })

comparison_results = pd.DataFrame(comparison_result_rows)
ranked_results = comparison_results.sort_values(
    "oos_mean_raw_R_predictive_ll", ascending=False, na_position="last"
)
with pd.option_context("display.max_colwidth", None):
    display(ranked_results)

# Implementation check: Normal(log R) and LogNormal(R) define the same raw-R density.
jacobian_pair = comparison_results.set_index("model").reindex(
    [CandidateName.NORMAL_LOG_R.value, CandidateName.LOGNORMAL_RAW_R.value]
)
if jacobian_pair["status"].eq("ok").all():
    jacobian_gap = abs(float(
        jacobian_pair["oos_mean_raw_R_predictive_ll"].iloc[0] - jacobian_pair["oos_mean_raw_R_predictive_ll"].iloc[1]
    ))
    print(f"|Normal(log R) - LogNormal(R)| OOS mean raw-R log-likelihood: {jacobian_gap:.2e}")
    assert jacobian_gap < JACOBIAN_CHECK_TOLERANCE, (
        "Normal(log R) and LogNormal(R) should score the same on the raw-R scale; "
        "check the Jacobian handling or whether one fit reached a different local optimum."
    )
else:
    print(f"Jacobian check skipped; candidate status: {jacobian_pair['status'].to_dict()}")

,model,distribution,observation,status,train_mean_raw_R_predictive_ll,oos_mean_raw_R_predictive_ll,oos_p05_raw_R_predictive_ll,oos_sequence_order_premium,slow_R_center,fast_R_center,center_ratio,train_fast_share,oos_fast_share,train_mean_confidence,oos_mean_confidence,train_flip_rate,oos_flip_rate,slow_R_self_transition,fast_R_self_transition,error
3,lognormal_raw_R,LOGNORMAL,raw_R,ok,6.864923,7.121507,4.661046,0.075406,0.000253,0.001686,6.651456,0.191000,0.089263,0.940673,0.950031,0.061528,0.059585,0.971322,0.884992,
0,normal_log_R,NORMAL,log_R,ok,6.864923,7.121507,4.661046,0.075406,0.000253,0.001686,6.651456,0.191000,0.089263,0.940673,0.950031,0.061528,0.059585,0.971322,0.884992,
2,lambda_log_R,LAMDA,log_R,ok,6.809704,7.071547,4.598169,0.042179,0.000265,0.001444,5.439852,0.219165,0.100906,0.928811,0.934256,0.054728,0.046632,0.986732,0.957967,
4,gamma_raw_R,GAMMA,raw_R,ok,6.825000,7.070797,4.631640,0.086673,0.000249,0.001837,7.363301,0.172871,0.093144,0.950444,0.961115,0.067358,0.075130,0.969576,0.862310,
1,studentt_log_R,STUDENTT,log_R,ok,6.811448,7.068157,4.630425,0.049460,0.000256,0.001383,5.399060,0.233085,0.120310,0.916242,0.918327,0.065091,0.054404,0.979599,0.942939,
5,exponential_raw_R,EXPONENTIAL,raw_R,ok,6.811735,7.046066,4.641013,0.075483,0.000259,0.001713,6.612672,0.179994,0.097025,0.940750,0.949199,0.053109,0.056995,0.978605,0.910817,


|Normal(log R) - LogNormal(R)| OOS mean raw-R log-likelihood: 2.97e-09


### Reading the comparison

Prefer higher `oos_mean_raw_R_predictive_ll`, but require stability across chronological folds and seeds before choosing a production family; the table at the end of this notebook provides that check. The fifth-percentile score exposes tail failures that the mean can hide. `oos_sequence_order_premium > 0` means the actual event ordering is more predictable than shuffled ordering; near zero means the result behaves mostly like a mixture model on the marginal R distribution.

Normal-on-log-R and LogNormal-on-raw-R should appear as a tied pair: they are the same model. A gap between them is an implementation error, not evidence.

The centers and labels are semantic summaries: `slow_R` is always the state with the lower probability-weighted training mean of raw R (`semantic_r_state_order`, the rule shared with `1_threshold_dc.ipynb` and Chapter 3). They must not be used to select the winning model merely because one produces a visually pleasing orange share.

In [15]:
if not comparison_label_frames:
    raise RuntimeError("Every candidate HMM failed; inspect the error column above.")

comparison_event_labels = pd.concat(comparison_label_frames, ignore_index=True)
successful_models = comparison_event_labels["model"].drop_duplicates().tolist()
full_label_index = pd.MultiIndex.from_product(
    [successful_models, ["train", "oos"], ["slow_R", "fast_R"]],
    names=["model", "sample", "state_name"],
)
label_summary = (
    comparison_event_labels.groupby(["model", "sample", "state_name"]).size()
    .reindex(full_label_index, fill_value=0).rename("event_count").reset_index()
)
label_summary["share"] = label_summary["event_count"] / label_summary.groupby(
    ["model", "sample"]
)["event_count"].transform("sum")

display(label_summary)
plot_hmm_model_label_shares(label_summary)

,model,sample,state_name,event_count,share
0,normal_log_R,train,slow_R,2499,0.809000
1,normal_log_R,train,fast_R,590,0.191000
2,normal_log_R,oos,slow_R,704,0.910737
3,normal_log_R,oos,fast_R,69,0.089263
4,studentt_log_R,train,slow_R,2369,0.766915
5,studentt_log_R,train,fast_R,720,0.233085
6,studentt_log_R,oos,slow_R,680,0.879690
7,studentt_log_R,oos,fast_R,93,0.120310
8,lambda_log_R,train,slow_R,2412,0.780835
9,lambda_log_R,train,fast_R,677,0.219165


In [16]:
oos_label_matrix = (
    comparison_event_labels.loc[comparison_event_labels["sample"] == "oos"]
    .pivot(index="event_id", columns="model", values="state_name")
)
model_names = oos_label_matrix.columns.tolist()
raw_agreement = pd.DataFrame(index=model_names, columns=model_names, dtype=float)
kappa_agreement = pd.DataFrame(index=model_names, columns=model_names, dtype=float)
for left_model in model_names:
    for right_model in model_names:
        left_labels = oos_label_matrix[left_model]
        right_labels = oos_label_matrix[right_model]
        valid = left_labels.notna() & right_labels.notna()
        raw_agreement.loc[left_model, right_model] = float(
            np.mean(left_labels.loc[valid].to_numpy() == right_labels.loc[valid].to_numpy())
        )
        kappa_agreement.loc[left_model, right_model] = cohen_kappa_score(
            left_labels.loc[valid], right_labels.loc[valid], labels=["slow_R", "fast_R"]
        )

display(raw_agreement.round(3))
plot_hmm_label_agreement(raw_agreement, title="OOS hard-label agreement")
display(kappa_agreement.round(3))
plot_hmm_label_agreement(kappa_agreement, title="OOS Cohen's kappa (base-rate adjusted)")

,exponential_raw_R,gamma_raw_R,lambda_log_R,lognormal_raw_R,normal_log_R,studentt_log_R
exponential_raw_R,1.000,0.975,0.939,0.974,0.974,0.940
gamma_raw_R,0.975,1.000,0.922,0.975,0.975,0.924
lambda_log_R,0.939,0.922,1.000,0.947,0.947,0.981
lognormal_raw_R,0.974,0.975,0.947,1.000,1.000,0.946
normal_log_R,0.974,0.975,0.947,1.000,1.000,0.946
studentt_log_R,0.940,0.924,0.981,0.946,0.946,1.000


,exponential_raw_R,gamma_raw_R,lambda_log_R,lognormal_raw_R,normal_log_R,studentt_log_R
exponential_raw_R,1.000,0.857,0.659,0.847,0.847,0.693
gamma_raw_R,0.857,1.000,0.557,0.852,0.852,0.600
lambda_log_R,0.659,0.557,1.000,0.692,0.692,0.901
lognormal_raw_R,0.847,0.852,0.692,1.000,1.000,0.711
normal_log_R,0.847,0.852,0.692,1.000,1.000,0.711
studentt_log_R,0.693,0.600,0.901,0.711,0.711,1.000


In [17]:
plot_hmm_model_event_strips(comparison_event_labels, sample="oos")

In [18]:
bar_model_labels = pd.DataFrame(index=prices.index)
for model_name, model_labels in comparison_event_labels.groupby("model", sort=False):
    event_updates = (
        model_labels.sort_values("information_time")
        .drop_duplicates("information_time", keep="last")
        .set_index("information_time")["state_name"]
    )
    bar_model_labels[model_name] = event_updates.reindex(prices.index).ffill()

plot_hmm_model_price_comparison(prices, bar_model_labels, n_prices=COMPARISON_PRICE_BARS)

## Stability across chronological folds and seeds

One split and one seed cannot show whether a ranking is stable. This section refits every candidate:

1. **Expanding chronological folds.** Train on the first 60%, 70%, and 80% of events and score the block that follows (60–70%, 70–80%, 80–100%). The 80% fold is the main comparison above. Scores from different folds come from different periods and block lengths, so compare **ranks within a fold**, not raw scores across folds.
2. **Seeds on the main split.** Refit with extra random seeds, each with the same `N_RESTARTS`. A spread across seeds means EM lands in different local optima.

Read ranks within each fold, never raw scores across folds. Two candidates whose scores differ by less than the largest seed-to-seed spread on the main split are treated as tied and share an average rank, so numerical noise cannot order them. A family leads only if it is in the top group in every fold; the cell prints that conclusion. `collapsed` means one state took every event in a sample. This is the most expensive cell in the notebook (one fit per candidate per extra fold and seed); set `RUN_STABILITY_GRID = False` in the configuration cell to skip it.

In [19]:
def stability_row(spec: HmmCandidateSpec, train_fraction: float, test_end: int, seed: int) -> dict:
    train_count = int(len(comparison_events) * train_fraction)
    row = {"model": spec.name.value, "fold_train_fraction": train_fraction, "seed": seed}
    try:
        model = fit_candidate(spec, train_count, seed)
        observations = observations_for(spec)[:test_end]
        scores = candidate_raw_r_scores(model, spec, observations, r_all[:test_end])
        probabilities = model.predict_proba(observations)
        _, _, semantic_probabilities = semantic_fast_flags(probabilities, train_count)
        is_fast = np.argmax(semantic_probabilities, axis=1) == 1
        train_fast_share = float(is_fast[:train_count].mean())
        test_fast_share = float(is_fast[train_count:test_end].mean())
        collapsed = train_fast_share in {0.0, 1.0} or test_fast_share in {0.0, 1.0}
        row.update({
            "status": "collapsed" if collapsed else "ok",
            "test_events": test_end - train_count,
            "test_mean_raw_R_predictive_ll": float(scores[train_count:test_end].mean()),
            "train_fast_share": train_fast_share,
            "test_fast_share": test_fast_share,
        })
    except Exception as exc:
        row.update({"status": "failed", "test_mean_raw_R_predictive_ll": np.nan, "error": repr(exc)})
    return row


stability_rows = []
for result in comparison_result_rows:
    stability_rows.append({
        "model": result["model"],
        "fold_train_fraction": TRAIN_FRACTION,
        "seed": RANDOM_STATE,
        "status": result["status"],
        "test_events": len(comparison_events) - train_event_count,
        "test_mean_raw_R_predictive_ll": result["oos_mean_raw_R_predictive_ll"],
        "train_fast_share": result.get("train_fast_share", np.nan),
        "test_fast_share": result.get("oos_fast_share", np.nan),
    })

if RUN_STABILITY_GRID:
    fold_ends = list(FOLD_TRAIN_FRACTIONS[1:]) + [1.0]
    for spec in CANDIDATES:
        for train_fraction, end_fraction in zip(FOLD_TRAIN_FRACTIONS, fold_ends, strict=True):
            if train_fraction == TRAIN_FRACTION:
                continue  # already scored by the main comparison
            test_end = len(comparison_events) if end_fraction >= 1.0 else int(len(comparison_events) * end_fraction)
            stability_rows.append(stability_row(spec, train_fraction, test_end, RANDOM_STATE))
        for seed in EXTRA_STABILITY_SEEDS:
            stability_rows.append(stability_row(spec, TRAIN_FRACTION, len(comparison_events), seed))

stability = pd.DataFrame(stability_rows)

fold_view = stability.loc[stability["seed"].eq(RANDOM_STATE)]
fold_scores = fold_view.pivot(index="model", columns="fold_train_fraction", values="test_mean_raw_R_predictive_ll")
fold_status = fold_view.pivot(index="model", columns="fold_train_fraction", values="status")

seed_view = stability.loc[stability["fold_train_fraction"].eq(TRAIN_FRACTION)]
seed_summary = seed_view.groupby("model").agg(
    seeds=("seed", "nunique"),
    collapsed_runs=("status", lambda status: int(status.eq("collapsed").sum())),
    failed_runs=("status", lambda status: int(status.eq("failed").sum())),
    oos_ll_min=("test_mean_raw_R_predictive_ll", "min"),
    oos_ll_max=("test_mean_raw_R_predictive_ll", "max"),
    oos_fast_share_min=("test_fast_share", "min"),
    oos_fast_share_max=("test_fast_share", "max"),
)
seed_summary["oos_ll_range"] = seed_summary["oos_ll_max"] - seed_summary["oos_ll_min"]

# Scores closer than the largest seed-to-seed spread are ties: rank them together.
TIE_TOLERANCE = max(float(seed_summary["oos_ll_range"].max(skipna=True)), 1e-6)


def tolerant_ranks(scores: pd.Series) -> pd.Series:
    ordered = scores.dropna().sort_values(ascending=False)
    groups, group, previous = [], 0, None
    for value in ordered:
        if previous is not None and previous - value > TIE_TOLERANCE:
            group += 1
        groups.append(group)
        previous = value
    plain = pd.Series(np.arange(1, len(ordered) + 1, dtype=float), index=ordered.index)
    ranks = plain.groupby(pd.Series(groups, index=ordered.index)).transform("mean")
    return ranks.reindex(scores.index)


fold_ranks = fold_scores.apply(tolerant_ranks, axis=0)
top_groups = {
    fold: tuple(sorted(fold_ranks.index[fold_ranks[fold].eq(fold_ranks[fold].min())]))
    for fold in fold_ranks.columns
}

print("Mean raw-R predictive log-likelihood of each fold's test block (columns = training fraction):")
display(fold_scores.round(4))
print(f"Rank within each fold (1 = best; scores within {TIE_TOLERANCE:.1e} share an average rank):")
display(fold_ranks)
for fold, group in top_groups.items():
    print(f"fold trained on {fold:.0%}: top group {group}")
always_top = set.intersection(*(set(group) for group in top_groups.values()))
print("Leads in every fold:", sorted(always_top) if always_top else "no family; there is no stable ranking")
display(fold_status)
print("Spread across seeds on the main split:")
display(seed_summary.sort_values("oos_ll_max", ascending=False).style.format({
    "oos_ll_min": "{:.4f}",
    "oos_ll_max": "{:.4f}",
    "oos_ll_range": "{:.2e}",
    "oos_fast_share_min": "{:.2%}",
    "oos_fast_share_max": "{:.2%}",
}))

Mean raw-R predictive log-likelihood of each fold's test block (columns = training fraction):


fold_train_fraction,0.6,0.7,0.8
model,,,
exponential_raw_R,7.3095,6.8921,7.0461
gamma_raw_R,7.3145,6.9057,7.0708
lambda_log_R,7.3108,6.8897,7.0715
lognormal_raw_R,7.3693,6.9384,7.1215
normal_log_R,7.3693,6.9384,7.1215
studentt_log_R,7.3216,6.8915,7.0682


Rank within each fold (1 = best; scores within 4.5e-03 share an average rank):


fold_train_fraction,0.6,0.7,0.8
model,,,
exponential_raw_R,5.0,5.0,6.0
gamma_raw_R,5.0,3.0,4.0
lambda_log_R,5.0,5.0,4.0
lognormal_raw_R,1.5,1.5,1.5
normal_log_R,1.5,1.5,1.5
studentt_log_R,3.0,5.0,4.0


fold trained on 60%: top group ('lognormal_raw_R', 'normal_log_R')
fold trained on 70%: top group ('lognormal_raw_R', 'normal_log_R')
fold trained on 80%: top group ('lognormal_raw_R', 'normal_log_R')
Leads in every fold: ['lognormal_raw_R', 'normal_log_R']


fold_train_fraction,0.6,0.7,0.8
model,,,
exponential_raw_R,ok,ok,ok
gamma_raw_R,ok,ok,ok
lambda_log_R,ok,ok,ok
lognormal_raw_R,ok,ok,ok
normal_log_R,ok,ok,ok
studentt_log_R,ok,ok,ok


Spread across seeds on the main split:


,seeds,collapsed_runs,failed_runs,oos_ll_min,oos_ll_max,oos_fast_share_min,oos_fast_share_max,oos_ll_range
model,,,,,,,,
normal_log_R,3,0,0,7.1215,7.1215,8.93%,8.93%,2.47e-06
lognormal_raw_R,3,0,0,7.1215,7.1215,8.93%,8.93%,3.59e-06
lambda_log_R,3,0,0,7.0715,7.0716,10.09%,10.09%,1.34e-05
gamma_raw_R,3,0,0,7.0663,7.0708,8.80%,9.31%,4.46e-03
studentt_log_R,3,0,0,7.0681,7.0682,12.03%,12.03%,1.07e-05
exponential_raw_R,3,0,0,7.0461,7.0462,9.70%,9.83%,1.28e-04
